# Part 4: Dataset Generation

Now that the pipeline has created all feature and outcome tables, we'll generate analysis-ready CSV datasets.

---

## What Are Datasets?

Datasets are CSV files that combine:
- **Features** from specific time windows (W1, W6, W24)
- **Outcomes** (mortality, deterioration, cardiac events)
- **Demographics** (age, gender, etc.)
- **Optional ECG features** (if available)

These are ready for machine learning, statistical analysis, or data exploration.

---

## Setup

In [ ]:
cd c:\Users\Lab\Desktop\MIMICc_Deterioration_Pipeline

c:\Users\Lab\Desktop\Cardiac_Deterioration_Pipeline


In [ ]:
import os
os.environ['PGPASSWORD'] = "---"
import pandas as pd
from pathlib import Path
from src.utils import load_yaml
from src.db import get_conn

# Load config
cfg = load_yaml("config/config.yaml")

# Set password if needed
if 'PGPASSWORD' not in os.environ:
    import getpass
    os.environ['PGPASSWORD'] = getpass.getpass("Enter PostgreSQL password: ")

# Create output directory
output_dir = Path("artifacts/datasets")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Output directory: {output_dir}")

✓ Output directory: artifacts\datasets


---

## Option 1: Generate Standard Datasets

Generate predefined datasets using the standard pipeline:

In [8]:
# List available predefined datasets
from src.make_datasets import list_available_datasets

list_available_datasets()


Available Dataset Specifications:
  ed_w6_det24_admitted                W6   deterioration_24h         ✓ ECG
  ed_w6_det24_all                     W6   deterioration_24h         ✓ ECG
  ed_w1_det24_admitted                W1   deterioration_24h         ✓ ECG
  ed_w6_ward_failure_admitted         W6   ward_failure_24_48h       ✓ ECG
  ed_w6_coronary72_admitted           W6   coronary_event_72h        ✓ ECG
  ed_w6_acs72_admitted                W6   acs_72h                   ✓ ECG
  ed_w6_revasc72_admitted             W6   revasc_72h                ✓ ECG
  ed_w6_icu24_admitted                W6   icu_24h                   ✓ ECG
  ed_w6_death24_admitted              W6   death_24h                 ✓ ECG
  ed_w6_cardiac_arrest_admitted       W6   cardiac_arrest_24h        ✓ ECG
  ed_w6_vent24_admitted               W6   vent_24h                  ✓ ECG
  ed_w6_det24_no_ecg                  W6   deterioration_24h           ---
  ed_w24_det48_admitted               W24  deterioration_48h     

In [ ]:
# Generate ALL standard datasets
# This will create ~10-15 datasets (may take 5-10 minutes)

from src.make_datasets import generate_all_datasets

print("Generating all standard datasets...\n")
datasets = generate_all_datasets(cfg, output_dir=str(output_dir))

print(f"\n✓ Generated {len(datasets)} datasets in {output_dir}")

---

## Option 2: Generate Specific Dataset

Create a single custom dataset with your choice of window and outcome:

In [12]:
from src.materialize_datasets import materialize_dataset

conn = get_conn(cfg)

# Example: W6 features with 24h mortality, admitted patients only
df = materialize_dataset(
    conn=conn,
    cfg=cfg,
    window='W6',              # Options: W1, W6, W24
    outcome_col='death_24h',           # Any outcome from tmp_outcomes
    cohort_type='admitted',          # Options: all, admitted, not_admitted
    include_ecg=False,                 # Set True to include ECG features
    out_csv=str(output_dir / 'my_w6_mortality_dataset.csv')
)

print(f"✓ Dataset created: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"  Outcome rate: {df['y'].mean():.2%}")
print(f"  Saved to: {output_dir / 'my_w6_mortality_dataset.csv'}")

conn.close()

# Preview
df.head()

⚠️  Very low outcome rate (0.29%). May cause training issues.


✓ Dataset created: 202,990 rows × 63 columns
  Outcome rate: 0.29%
  Saved to: artifacts\datasets\my_w6_mortality_dataset.csv


,stay_id,subject_id,hadm_id,ed_intime,ed_outtime,age_at_ed,gender,ed_los_hours,arrival_transport,race,...,missing_creatinine_first_6h,missing_potassium_first_6h,missing_sodium_first_6h,missing_bicarbonate_first_6h,missing_wbc_first_6h,missing_hemoglobin_first_6h,missing_platelet_first_6h,missing_time_to_first_lab_hours,missing_time_to_first_med_hours,missing_sbp_cv_6h
0,30000012,11714491,21562392,2126-02-14 20:22:00,2126-02-15 01:59:00,66,F,5.6166666666666667,AMBULANCE,WHITE,...,0,0,0,0,0,0,0,0,0,0
1,30000038,13821532,26255538,2152-12-07 16:37:00,2152-12-07 19:55:00,82,F,3.3000000000000000,AMBULANCE,WHITE,...,1,1,1,1,1,1,1,1,0,0
2,30000039,13340997,23100190,2165-10-06 11:47:00,2165-10-06 20:18:00,80,M,8.5166666666666667,WALK IN,WHITE,...,1,1,1,1,1,1,1,1,1,0
3,30000177,17937834,23831044,2143-12-27 22:50:00,2143-12-28 03:48:00,38,M,4.9666666666666667,WALK IN,ASIAN - SOUTH EAST ASIAN,...,0,0,0,0,0,0,0,0,0,1
4,30000204,11615015,25540031,2132-10-10 06:36:00,2132-10-10 18:45:00,45,M,12.1500000000000000,AMBULANCE,WHITE,...,1,1,1,1,1,1,1,1,0,0


---

## Option 3: Generate Hybrid Dataset (Multi-Window, Multi-Outcome)

Create advanced datasets combining multiple windows and/or outcomes:

### Example 1: Multi-Mortality Dataset (W6 + W24)

In [ ]:
from src.generate_advanced_dataset import generate_advanced_dataset

# W6 + W24 features with multiple mortality outcomes
df = generate_advanced_dataset(
    windows=['W6', 'W24'],
    outcome_cols=['death_24h', 'death_48h', 'death_hosp'],
    cohort_type='admitted',
    dataset_name='w6_w24_multi_mortality',
    output_dir=str(output_dir),
    generate_reports=True
)

print(f"\n✓ Hybrid dataset created")
print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"\nOutcome rates:")
for outcome in ['death_24h', 'death_48h', 'death_hosp']:
    col = f'y_{outcome}'
    if col in df.columns:
        print(f"  {outcome:15s}: {df[col].mean():.2%}")


DATA QUALITY REPORT: w6_w24_multi_mortality

📊 Basic Info:
   Rows: 202,990
   Columns: 115
   Memory: 819.26 MB
Missing Values:
   Overall: 17.5%
   Columns with missing: 60/115
Sanity Checks:
   ✓ unique_stay_ids: All stay_ids unique
   ✓ no_all_null_columns: No all-null columns found

⚠️  Warnings (3):
   - Column 'is_hs_troponin_6h' has constant value: 1.0
   - Column 'is_hs_troponin_24h' has constant value: 1.0
   - 17 columns have >50% missing values
Quality Score: 85.3/100

✓ Hybrid dataset created
  Shape: 202,990 rows × 115 columns

Outcome rates:
  death_24h      : 0.29%
  death_48h      : 0.55%
  death_hosp     : 2.26%


### Example 2: Deterioration Outcomes

In [ ]:
# W6 features with multiple deterioration outcomes
df = generate_advanced_dataset(
    windows=['W6'],
    outcome_cols=['deterioration_24h', 'icu_24h', 'pressor_24h', 'vent_24h'],
    cohort_type='all',
    dataset_name='w6_deterioration_outcomes',
    output_dir=str(output_dir)
)

print(f"✓ Dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

### Example 3: Cardiac Events with ECG

In [ ]:
# W1 + W6 features with cardiac outcomes and ECG
df = generate_advanced_dataset(
    windows=['W1', 'W6'],
    outcome_cols=['acs_hosp', 'cardiac_arrest_hosp', 'revasc_hosp'],
    cohort_type='admitted',
    include_ecg=True,
    dataset_name='w1_w6_cardiac_with_ecg',
    output_dir=str(output_dir)
)

print(f"✓ Dataset with ECG: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

✓ Dataset with ECG: 202,990 rows × 144 columns


In [23]:
from src.build_ecg_features import validate_ecg_features
from src.db import get_conn
from src.utils import load_yaml
import logging

# Setup logging to display in notebook
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Load configuration
cfg = load_yaml("config/config.yaml")

# Establish database connection
conn = get_conn(cfg)

print("=" * 70)
print("ECG FEATURES VALIDATION - W6 WINDOW")
print("=" * 70)

# Validate ECG features for W6 window
validate_ecg_features(conn, cfg, window="W6")

print("\n" + "=" * 70)
print("ECG FEATURES VALIDATION - W1 WINDOW")
print("=" * 70)

# Also validate W1 for comparison
validate_ecg_features(conn, cfg, window="W1")

# Close the connection
conn.close()

print("\n✓ ECG validation complete")

Loaded configuration from: config/config.yaml
Connected to database: mimiciv @ localhost
Validating ECG features W6...
  Total stays: 424,952
  With ECG: 145,790 (34.3%)
  HR: mean=81.3, median=78.0, range=[0.9, 253.2]


ECG FEATURES VALIDATION - W6 WINDOW


  QRS duration: mean=116.5ms, median=93.0ms
Validating ECG features W1...
  Total stays: 424,952
  With ECG: 112,948 (26.6%)
  HR: mean=82.1, median=79.1, range=[0.9, 253.2]
  QRS duration: mean=117.3ms, median=93.0ms



ECG FEATURES VALIDATION - W1 WINDOW

✓ ECG validation complete


---

## Option 4: Custom SQL Query

For maximum flexibility, write a custom SQL query:

In [27]:
conn = get_conn(cfg)

# Example: Elderly patients (65+) with elevated troponin
query = """
SELECT
    c.stay_id,
    c.subject_id,
    c.age_at_ed,
    c.gender,
    
    -- W6 features
    f.sbp_mean_6h,
    f.hr_mean_6h,
    f.rr_mean_6h,
    f.spo2_min_6h,
    f.lactate_first_6h,
    f.troponin_first_6h,
    f.creatinine_first_6h,
    
    -- Outcomes
    o.death_24h,
    o.death_48h,
    o.death_hosp,
    o.icu_24h
    
FROM tmp_base_ed_cohort c
JOIN tmp_features_w6 f ON c.stay_id = f.stay_id
JOIN tmp_ed_outcomes o ON c.stay_id = o.stay_id
WHERE 
    c.was_admitted = 1
    AND c.age_at_ed >= 65
    AND f.troponin_first_6h > 0.04  -- Elevated troponin threshold
"""

df = pd.read_sql(query, conn)
conn.close()

print(f"Custom cohort: {len(df):,} patients")
print(f"Mean age: {df['age_at_ed'].mean():.1f}")
print(f"Hospital mortality: {df['death_hosp'].mean():.2%}")

# Save
output_path = output_dir / 'elderly_elevated_troponin.csv'
df.to_csv(output_path, index=False)
print(f"\n✓ Saved to: {output_path}")

df.head()

Connected to database: mimiciv @ localhost


Custom cohort: 1,932 patients
Mean age: 79.2
Hospital mortality: 14.91%

✓ Saved to: artifacts\datasets\elderly_elevated_troponin.csv


C:\Users\Lab\AppData\Local\Temp\ipykernel_6620\2927133049.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,stay_id,subject_id,age_at_ed,gender,sbp_mean_6h,hr_mean_6h,rr_mean_6h,spo2_min_6h,lactate_first_6h,troponin_first_6h,creatinine_first_6h,death_24h,death_48h,death_hosp,icu_24h
0,30005519,15375544,93.0,M,149.0,101.0,22.000000,94.0,NaN,0.28,1.0,0,0,0,0
1,30005663,13495822,83.0,F,116.0,72.0,20.000000,100.0,NaN,0.26,2.0,0,0,0,0
2,30008125,18064328,91.0,F,NaN,NaN,NaN,NaN,1.6,0.05,0.8,0,0,1,1
3,30008190,15357459,77.0,M,106.0,81.0,18.666667,97.0,179.0,0.06,2.1,0,0,0,0
4,30016991,16320291,65.0,M,144.4,88.4,20.400000,97.0,NaN,0.17,NaN,0,0,0,1


---

## View Generated Datasets

In [28]:
import os

# List all generated datasets
datasets = list(output_dir.glob('*.csv'))

print(f"Generated datasets in {output_dir}:\n")
print(f"{'Filename':<50s} {'Size':>15s}")
print("-" * 70)

for ds in sorted(datasets):
    size_mb = ds.stat().st_size / 1024 / 1024
    print(f"{ds.name:<50s} {size_mb:>12.1f} MB")

print(f"\nTotal datasets: {len(datasets)}")

Generated datasets in artifacts\datasets:

Filename                                                      Size
----------------------------------------------------------------------
ed_w1_icu24.csv                                            90.1 MB
ed_w24_det48.csv                                          153.0 MB
ed_w6_det24.csv                                           157.3 MB
elderly_elevated_troponin.csv                               0.1 MB
my_w6_mortality_dataset.csv                                78.4 MB
w1_w6_cardiac_with_ecg.csv                                136.4 MB
w6_w24_multi_mortality.csv                                128.4 MB

Total datasets: 7


---

## Quick Dataset Exploration

Load and explore one of your datasets:

In [29]:
# Load a dataset
# EDIT THIS: Choose your dataset file
dataset_file = 'my_w6_mortality_dataset.csv'  # Change this to your file

df = pd.read_csv(output_dir / dataset_file)

print(f"Dataset: {dataset_file}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns\n")

# Basic info
print("Column types:")
print(df.dtypes.value_counts())

print("\nMissing values (top 10):")
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)
print(missing_pct.sort_values(ascending=False).head(10))

# Preview
print("\nFirst 5 rows:")
df.head()

Dataset: my_w6_mortality_dataset.csv
Shape: 202,990 rows × 63 columns

Column types:
float64    30
int64      27
object      6
Name: count, dtype: int64

Missing values (top 10):
troponin_first_6h       97.681659
is_hs_troponin_6h       97.681659
lactate_first_6h        88.978275
platelet_first_6h       79.976846
bicarbonate_first_6h    79.769939
hemoglobin_first_6h     79.653678
creatinine_first_6h     79.530026
sodium_first_6h         79.294054
potassium_first_6h      79.107345
wbc_first_6h            78.891078
dtype: float64

First 5 rows:


,stay_id,subject_id,hadm_id,ed_intime,ed_outtime,age_at_ed,gender,ed_los_hours,arrival_transport,race,...,missing_creatinine_first_6h,missing_potassium_first_6h,missing_sodium_first_6h,missing_bicarbonate_first_6h,missing_wbc_first_6h,missing_hemoglobin_first_6h,missing_platelet_first_6h,missing_time_to_first_lab_hours,missing_time_to_first_med_hours,missing_sbp_cv_6h
0,30000012,11714491,21562392,2126-02-14 20:22:00,2126-02-15 01:59:00,66,F,5.616667,AMBULANCE,WHITE,...,0,0,0,0,0,0,0,0,0,0
1,30000038,13821532,26255538,2152-12-07 16:37:00,2152-12-07 19:55:00,82,F,3.300000,AMBULANCE,WHITE,...,1,1,1,1,1,1,1,1,0,0
2,30000039,13340997,23100190,2165-10-06 11:47:00,2165-10-06 20:18:00,80,M,8.516667,WALK IN,WHITE,...,1,1,1,1,1,1,1,1,1,0
3,30000177,17937834,23831044,2143-12-27 22:50:00,2143-12-28 03:48:00,38,M,4.966667,WALK IN,ASIAN - SOUTH EAST ASIAN,...,0,0,0,0,0,0,0,0,0,1
4,30000204,11615015,25540031,2132-10-10 06:36:00,2132-10-10 18:45:00,45,M,12.150000,AMBULANCE,WHITE,...,1,1,1,1,1,1,1,1,0,0


In [30]:
# Outcome statistics
outcome_cols = [col for col in df.columns if col.startswith('y')]

if outcome_cols:
    print("Outcome Statistics:\n")
    for col in outcome_cols:
        count = df[col].sum()
        rate = df[col].mean()
        print(f"{col:20s}: {count:>6,} ({rate:>6.2%})")
else:
    print("No outcome columns found (column names don't start with 'y')")

Outcome Statistics:

y                   :    595 ( 0.29%)


---

**✋ CHECKPOINT:** You should now have:
- ✓ Multiple CSV datasets in artifacts/datasets/
- ✓ Datasets with features, outcomes, and demographics
- ✓ Understanding of dataset structure

Continue to Part 5 for analysis examples!

---